In [6]:
import pandas as pd
from pathlib import Path
from datetime import datetime

NORMALIZED_PATH = Path("../data/normalized/2026-05-21_13-20-28.csv")
MART_DIR = Path("../data/mart")
MART_DIR.mkdir(parents=True, exist_ok=True)

REGIONS_PATH = Path("../reference/regions.csv")

df = pd.read_csv(NORMALIZED_PATH)
df['time'] = pd.to_datetime(df['time'])

regions = pd.read_csv(REGIONS_PATH)

df['region_id'] = 'US_CA'
df = df.merge(regions, on='region_id', how='left')

df['date'] = df['time'].dt.date

daily = df.groupby('date').agg(
    earthquake_count=('event_id', 'count'),
    avg_magnitude=('magnitude', 'mean'),
    max_magnitude=('magnitude', 'max'),
    min_magnitude=('magnitude', 'min'),
    avg_depth_km=('depth_km', 'mean'),
    max_depth_km=('depth_km', 'max')
).reset_index()

daily['date'] = pd.to_datetime(daily['date'])

daily['avg_magnitude_7d'] = daily['avg_magnitude'].rolling(window=7, min_periods=1).mean()
daily['earthquake_count_7d'] = daily['earthquake_count'].rolling(window=7, min_periods=1).sum()

daily['deep_events_count'] = df.groupby('date').apply(
    lambda x: (x['depth_km'] > 70).sum()
).reset_index(drop=True)

daily['deep_events_pct'] = (daily['deep_events_count'] / daily['earthquake_count'] * 100).round(1)

daily['year'] = daily['date'].dt.year
daily['month'] = daily['date'].dt.month
daily['week'] = daily['date'].dt.isocalendar().week
daily['day_of_week'] = daily['date'].dt.dayofweek

daily['region_id'] = 'US_CA'
daily['region_name'] = regions.loc[regions['region_id'] == 'US_CA', 'region_name'].values[0]

output_path = MART_DIR / f"mart_daily_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.csv"
daily.to_csv(output_path, index=False)